# R1-based LLM Implementation

Architecture of the R1-based LLM implementation:
* Input embeddings
* Multi-head latent attention
* Mixture of experts (MoE) layers
* Output projection
* Multi-token prediction (integrated into transformer architecture)
* Decoupled Rotary Positional Embeddings (Integrated into attention mechanism)

## Import Libraries

In [2]:
import json
import urllib.request
import os
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Any, Tuple, List, Union, Dict, Optional

from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import Whitespace
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import Adam, AdamW
import math
from scipy.ndimage import uniform_filter1d
from matplotlib import pyplot as plt
from torchviz import make_dot

In [3]:
print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

PyTorch version: 2.10.0
CUDA available: False


## Data Preparation

### Download and prepare the dataset (e.g., tiny Shakespeare)

In [4]:
URL = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
FILE_NAME = "input.txt"
DATA_DIR = "data"
os.makedirs(DATA_DIR, exist_ok=True)

FULL_PATH = os.path.join(DATA_DIR, FILE_NAME)

# Download the dataset if it doesn't exist
if not os.path.exists(FULL_PATH):
    urllib.request.urlretrieve(URL, FULL_PATH)

print("Dataset downloaded to:", FULL_PATH)

with open(FULL_PATH, 'r', encoding='utf-8') as f:
    text = f.read()
    print("Dataset length (characters):", len(text))

Dataset downloaded to: data/input.txt
Dataset length (characters): 1115394


###  Tokenization and Vocabulary Building

In [5]:
VOCAB_SIZE = int(math.sqrt(len(text)))
TOKENIZER_SAVE_PATH = os.path.join(DATA_DIR, "bpe_tokenizer.json")

def train_bpe_tokenizer(text: str, vocab_size:int, save_path: str) -> Tokenizer:
    tokenizer = Tokenizer(BPE(unk_token="[UNK]"))
    tokenizer.pre_tokenizer = Whitespace()

    trainer = BpeTrainer(
        vocab_size=vocab_size,
        special_tokens = ["[UNK]", "[PAD]", "[BOS]", "[EOS]"],
        show_progress=True
    )

    tokenizer.train_from_iterator([text], trainer=trainer)
    tokenizer.save(save_path)

    print("Tokenizer trained and saved to:", save_path)
    return tokenizer

In [6]:
tokenizer = train_bpe_tokenizer(text, VOCAB_SIZE, TOKENIZER_SAVE_PATH)





Tokenizer trained and saved to: data/bpe_tokenizer.json


### Create PyTorch Dataset and DataLoader

In [7]:
class CustomDataset(Dataset):
    def __init__(self, text: str, tokenizer: Tokenizer, seq_length: int = 128, train: bool = True, train_split: float = 0.8):
        self.text = text
        self.tokenizer = tokenizer
        self.seq_length = seq_length
        self.train = train
        self.train_split = train_split

        # Encode the entire text
        encoded_text = self.tokenizer.encode(self.text)
        self.tokens = torch.tensor(encoded_text.ids, dtype=torch.long)

        # Split into train and validation sets
        split_idx = int(len(self.tokens) * self.train_split)
        if self.train:
            self.tokens = self.tokens[:split_idx]
        else:
            self.tokens = self.tokens[split_idx:]

        print("Dataset initialized. Total tokens:", len(self.tokens))

    def __len__(self):
        return max(0, len(self.tokens) - self.seq_length)

    def __getitem__(self, idx):
        return self.tokens[idx:idx+self.seq_length], self.tokens[idx+1:idx+self.seq_length+1]


In [8]:
# Create train and validation datasets
SEQ_LENGTH = 128
BATCH_SIZE = 32

train_dataset = CustomDataset(text, tokenizer, seq_length=SEQ_LENGTH, train=True)
val_dataset = CustomDataset(text, tokenizer, seq_length=SEQ_LENGTH, train=False)

# Create DataLoaders
# For training, we shuffle the data to ensure the model sees different sequences each epoch
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
# Note: For validation, we typically don't shuffle the data because we want to evaluate on the same sequence of data each epoch
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

Dataset initialized. Total tokens: 303268
Dataset initialized. Total tokens: 75817


## Transformer Architecture implementation with R1-based LLM Components

### Input Embeddings

In [9]:
class InputEmbeddings(nn.Module):
    """
    Token embedding layer that converts token IDs into dense vectors. This layer is essential for transforming discrete token IDs into continuous representations that can be processed by the transformer architecture. The embedding dimension is a hyperparameter that determines the size of the vector space in which the tokens will be represented.

    Args:
        vocab_size (int): Size of the vocabulary.
        d_model (int): Embedding dimension (size of the output vectors).
        padding_idx (int): Optional index token index.
        scale (bool): Whether to scale the embeddings by sqrt(d_model).
    """

    def __init__(
        self,
        vocab_size: int = VOCAB_SIZE,
        d_model: int = 512,
        padding_idx: int | None = None,
        scale: bool = True
    ) -> None:
        super(InputEmbeddings, self).__init__()
        self.embedding = nn.Embedding(vocab_size, d_model, padding_idx=padding_idx)
        self.scale = scale
        self.d_model = d_model

    def forward(self, input_ids: torch.Tensor) -> torch.Tensor:
        """
        Args:
            input_ids (torch.Tensor): Tensor of shape (batch_size, seq_length) containing token IDs.

        Returns:
            torch.Tensor: Tensor of shape (batch_size, seq_length, d_model) containing the embedded representations of the input tokens.
        """

        x = self.embedding(input_ids)

        if self.scale:
            x = x * math.sqrt(self.d_model)

        return x

### Decoupled Rotary Positional Embeddings

In [10]:
def rotate_half(x: torch.Tensor) -> torch.Tensor:
    """
    Splits the last dimension into two halves and rotates them:
    Args:
        x (torch.Tensor): Input tensor of shape (..., d_model).

    Returns:
        torch.Tensor: Rotated tensor of the same shape as input. [x1, x2] -> [-x2, x1]
    """

    half = x.size(-1) // 2
    x1 = x[..., :half]
    x2 = x[..., half:]
    return torch.cat([-x2, x1], dim=-1)

def apply_rotary_pos_emb(
        x: torch.Tensor,
        cos: torch.Tensor,
        sin: torch.Tensor,
) -> torch.Tensor:
    """
    x: [B, H, T, D]
    cos: [1 or B, 1, T, D]
    sin: [1 or B, 1, T, D]
    """

    return (x * cos) + (rotate_half(x) * sin)

class RotaryEmbedding(nn.Module):
    """
    Standard RoPE helper.

    Produces cos/sin tensors that can be applied to:
        - decoupled RoPE queries: [B, H, T, D_rope]
        - decoupled RoPE keys: [B, 1, T, D_rope]
    """

    def __init__(self, dim: int, base: int = 10_000) -> None:
        super().__init__()

        if dim % 2 != 0:
            raise ValueError("RoPE dimension must be even, got {}".format(dim))

        inv_freq = 1.0 / (base ** (torch.arange(0, dim, 2).float() / dim))

        self.register_buffer("inv_freq", inv_freq, persistent=False)
        self.dim = dim

    def get_cos_sin(
            self,
            seq_len: int,
            *,
            device: torch.device,
            dtype: torch.dtype,
            start_pos: int = 0
            ) -> Tuple[torch.Tensor, torch.Tensor]:

        """
            Computes the cosine and sine positional embeddings for a given sequence length.

            The method generates a range of positions starting from `start_pos` to `start_pos + seq_len - 1`, computes the outer product with the inverse frequencies, and then applies the cosine and sine functions to produce the positional embeddings.

            Args:
                seq_len (int): The length of the sequence for which to compute the positional embeddings.
                device (torch.device): The device on which to create the tensors (e.g., CPU or GPU).
                dtype (torch.dtype): The data type of the output tensors (e.g., torch.float32).
                start_pos (int, optional): The starting position index for the sequence. Defaults to 0.

            Returns:
                Tuple[torch.Tensor, torch.Tensor]: A tuple containing two tensors:
                    - cos: A tensor of shape (1, 1, seq_len, dim) containing the cosine positional embeddings.
                    - sin: A tensor of shape (1, 1, seq_len, dim) containing the sine positional embeddings.
        """

        positions = torch.arange(
            start_pos, start_pos + seq_len, device=device, dtype=torch.float32
        )
        freqs = torch.outer(positions, self.inv_freq) # [T, D/2]
        emb = torch.cat([freqs, freqs], dim=-1) # [T, D]

        cos = emb.cos().to(dtype=dtype)[None, None, :, :] # [1, 1, T, D]
        sin = emb.sin().to(dtype=dtype)[None, None, :, :] # [1, 1, T, D]

        return cos, sin

### Multi-head Latent Attention with Decoupled RoPE

In [11]:
class MultiHeadLatentAttention(nn.Module):
    """
    Multi-head latent attention mechanism that incorporates decoupled Rotary Positional Embeddings (RoPE). This attention mechanism allows the model to focus on different parts of the input sequence while also incorporating positional information through RoPE.

    Main idea:
      x
      ├─ Q latent branch:   x -> c_q  -> q_content
      │                     x -> c_q  -> q_rope
      └─ KV latent branch:  x -> c_kv -> k_content, v
                            x --------> shared k_rope

    Attention uses:
        q = [q_content, q_rope]
        k = [k_content, k_rope_shared]
        v = v_content

    Shapes:
        x: [B, T, D]
        out: [B, T, D]
    """

    def __init__(self, d_model: int, num_heads: int, head_dim: int, q_lora_rank: int, kv_lora_rank: int,
                 rope_head_dim: int, dropout: float = 0.0, bias: bool = False, rope_base: int = 10_000, *args: Any,
                 **kwargs: Any) -> None:

        super().__init__(*args, **kwargs)
        if rope_head_dim % 2 != 0:
            raise ValueError("RoPE head dimension must be even, got {}".format(rope_head_dim))

        if num_heads <= 0:
            raise ValueError("Number of heads must be positive, got {}".format(num_heads))

        if head_dim <= 0:
            raise ValueError("Head dimension must be positive, got {}".format(head_dim))

        self.d_model = d_model
        self.num_heads = num_heads
        self.head_dim = head_dim
        self.rope_head_dim = rope_head_dim
        self.q_lora_rank = q_lora_rank
        self.kv_lora_rank = kv_lora_rank
        self.total_qk_dim = head_dim + rope_head_dim
        self.scale = 1.0 / math.sqrt(self.total_qk_dim)

        # Query low-rank path
        self.w_dq = nn.Linear(d_model, q_lora_rank, bias=bias) # down projection
        self.w_uq = nn.Linear(q_lora_rank, num_heads * head_dim, bias=bias) # up projection for content queries

        self.w_qr = nn.Linear(q_lora_rank, num_heads * rope_head_dim, bias=bias) # RoPE queries

        # Key/Value low-rank path
        self.w_dkv = nn.Linear(d_model, kv_lora_rank, bias=bias) # latent KV
        self.w_uk = nn.Linear(kv_lora_rank, num_heads * head_dim, bias=bias)
        self.w_uv = nn.Linear(kv_lora_rank, num_heads * head_dim, bias=bias) # up projection for values

        # Shared RoPE for keys
        self.w_kr = nn.Linear(d_model, rope_head_dim, bias=bias)

        # output projection
        self.w_o = nn.Linear(num_heads * head_dim, d_model, bias=bias)

        self.attn_dropout = nn.Dropout(dropout)
        self.rope =  RotaryEmbedding(rope_head_dim, base = rope_base)

    def _reshape_heads(self, x: torch.Tensor, head_dim: int) -> torch.Tensor:
        """
        Reshapes the input tensor for multi-head attention.

        Args:
            x (torch.Tensor): Input tensor of shape (batch_size, seq_length, num_heads * head_dim).
            head_dim (int): The dimension of each attention head.

        Returns:
            torch.Tensor: Reshaped tensor of shape (batch_size, num_heads, seq_length, head_dim).
        """
        batch_size, seq_length, _ = x.size()
        return x.view(batch_size, seq_length, self.num_heads, head_dim).transpose(1, 2).contiguous()

    def forward(
            self,
            x: torch.Tensor,
            attention_mask: torch.Tensor | None = None,
            *,
            is_casual: bool = True,
            start_pos: int = 0,
            return_attn_weights: bool = False
                ):
        """
        Args:
            x: [B, T, D]
            attention_mask:
                Optional mask for valid keys.
                Supported simple form: [B, T], where 1/True = keep, 0/False = mask out.
            is_casual:
                Whether to apply causal masking. If True, the attention will be masked to prevent attending to future tokens.
            start_pos:
                Useful for generation offsets with RoPE positions.
            return_attn_weights:
                If True, returns attention probabilities.

        Returns:
            out: [B, T, D]
             (optional) attn_weights: [B, num_heads, T, T]
        """

        bsz, seq_len, _ = x.size()

        device = x.device
        dtype = x.dtype

        # Low-rank query path
        c_q = self.w_dq(x) # [B, T, q_lora_rank]
        q_content = self.w_uq(c_q) # [B, T, num_heads * head_dim]
        q_rope = self.w_qr(c_q) # [B, T, num_heads * rope_head_dim]

        q_content = self._reshape_heads(q_content, self.head_dim) # [B, num_heads, T, head_dim]
        q_rope = self._reshape_heads(q_rope, self.rope_head_dim)  # [B, num_heads, T, rope_head_dim]

        # Low-rank KV path
        c_kv = self.w_dkv(x)  # [B, T, kv_lora_rank]
        k_content = self.w_uk(c_kv) # [B, T, num_heads * head_dim]
        v = self.w_uv(c_kv) # [B, T, num_heads * head_dim]

        k_content = self._reshape_heads(k_content, self.head_dim) # [B, num_heads, T, head_dim]
        v = self._reshape_heads(v, self.head_dim) # [B, num_heads, T, head_dim]

        # Decoupled RoPE shared key
        k_rope_shared = self.w_kr(x) # [B, T, rope_head_dim]
        k_rope_shared = k_rope_shared.unsqueeze(1) # [B, 1, T, rope_head_dim]

        # Apply RoPE to the decoupled branches
        cos, sin = self.rope.get_cos_sin(seq_len, device=device, dtype=dtype, start_pos=start_pos)

        q_rope = apply_rotary_pos_emb(q_rope, cos, sin) # [B, num_heads, T, rope_head_dim]
        k_rope_shared = apply_rotary_pos_emb(k_rope_shared, cos, sin) # [B, 1, T, rope_head_dim]
        k_rope = k_rope_shared.expand(-1, self.num_heads, -1, -1) # [B, num_heads, T, rope_head_dim]

        # Combine content and RoPE parts
        q = torch.cat([q_content, q_rope], dim=-1) # [B, num_heads, T, head_dim + rope_head_dim]
        k = torch.cat([k_content, k_rope], dim=-1) # [B, num_heads, T, head_dim + rope_head_dim]

        attn_scores = torch.matmul(q, k.transpose(-2, -1)) * self.scale # [B, num_heads, T, T]

        # key padding mask: [B, T] -> [B, 1, 1, T]
        if attention_mask is not None:
            if attention_mask.dim() != 2 or attention_mask.shape != (bsz, seq_len):
                raise ValueError(f"Attention mask must be of shape {bsz, seq_len}, got {attention_mask.shape}")

            key_mask = attention_mask[:, None, None, :].to(dtype=torch.bool) # [B, 1, 1, T]
            attn_scores = attn_scores.masked_fill(
                ~key_mask,
                torch.finfo(attention_mask.dtype if attention_mask.is_floating_point() else x.dtype).min
                if x.dtype in (torch.float16, torch.bfloat16, torch.float32, torch.float64) else -1e9
            )

        # Causal mask
        if is_casual:
            casual_mask = torch.triu(
                torch.ones(seq_len, seq_len, device=device, dtype=torch.bool),
                diagonal=1
            )

            attn_scores = attn_scores.masked_fill(casual_mask[None, None, :,:],
                                                  torch.finfo(x.dtype).min if x.dtype in
                                                                              (torch.float16, torch.bfloat16, torch.float32, torch.float64) else -1e9
                                                  )


        # softmax in float32 for a bit more stability, then cast back
        attn_probs = F.softmax(attn_scores.float(), dim=-1).to(dtype)
        attn_probs = self.attn_dropout(attn_probs)

        # Values come only from the content/value branc
        out = torch.matmul(attn_probs, v) # [B, num_heads, T, head_dim]

        # Merge heads
        out = out.transpose(1, 2).contiguous().view(bsz, seq_len, self.num_heads * self.head_dim) # [B, T, num_heads * head_dim]
        out = self.w_o(out) # [B, T, D]

        if return_attn_weights:
            return out, attn_probs

        return out


### Mixture of Experts (MoE) Layer with auxiliary loss free balancing combined with specialized and shared experts

In [12]:

class SwiGLUExpert(nn.Module):
    """
    FFN expert:
        up + gate -> SiLU gate -> elementwise product -> down
    """

    def __init__(
            self,
            d_model: int,
            hidden_dim: int,
            dropout: float = 0.0,
            bias: bool = True
    ) -> None:
        super().__init__()

        if d_model <= 0:
            raise ValueError("Model dimension must be positive, got {}".format(d_model))

        if hidden_dim <= 0:
            raise ValueError("Hidden dimension must be positive, got {}".format(hidden_dim))

        self.up_proj = nn.Linear(d_model, hidden_dim, bias=bias)
        self.gate_proj = nn.Linear(d_model, hidden_dim, bias=bias)
        self.down_proj = nn.Linear(hidden_dim, d_model, bias=bias)
        self.dropout = nn.Dropout(dropout)


    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x: [B, T, D]

        Returns:
            out: [B, T, D]
        """

        gated = F.silu(self.gate_proj(x)) * self.up_proj(x) # [B, T, hidden_dim]
        gated = self.dropout(gated)

        return self.down_proj(gated) # [B, T, D]
class SharedExperts(nn.Module):
    """
    Always-on experts. Their outputs are summed.
    """

    def __init__(
            self,
            d_model: int,
            hidden_dim: int,
            num_shared_experts: int,
            dropout: float = 0.0,
            bias: bool = True
    ) -> None:
        super().__init__()

        if num_shared_experts <= 0:
            raise ValueError("Number of shared experts must be positive, got {}".format(num_shared_experts))

        self.experts = nn.ModuleList([
            SwiGLUExpert(d_model, hidden_dim, dropout=dropout, bias=bias)
            for _ in range(num_shared_experts)
        ])

    def forward(self, x: torch.Tensor) -> torch.Tensor:

        if len(self.experts) == 0:
            return torch.zeros_like(x)

        out = None

        for expert in self.experts:
            y = expert(x) # [B, T, D]
            out = y if out is None else out + y

        return out


@dataclass
class RouterStats:
    affinity_scores: torch.Tensor # [B, T, num_experts]
    biased_scores: torch.Tensor # [B, T, num_experts]
    top_k_indices: torch.Tensor # [B, T, k]
    top_k_gates: torch.Tensor # [B, T, k]
    expert_load: torch.Tensor # [num_experts]
    expert_bias: torch.Tensor # [num_experts]

class LossFreeBiasRouter(nn.Module):
    """
    Routed expert router.

    Key points:
        - affinity_score = sigmoid(router_linear(x))
        - routing decision uses affinity + expert_bias
        - final mixing weights are computed from the ORIGINAL affinity scores of selected experts, then normalized across the selected experts (not all experts)
        - expert_bias is updated after each step
    """

    def __init__(
            self,
            d_model: int,
            num_routed_experts: int,
            top_k: int,
            bias_update_speed: float = 1e-3,
    ) -> None:
        super().__init__()

        if num_routed_experts <= 0:
            raise ValueError("Number of routed experts must be positive, got {}".format(num_routed_experts))

        if top_k <= 0 or top_k > num_routed_experts:
            raise ValueError("top_k must be in the range [1, num_routed_experts], got {}".format(top_k))

        if bias_update_speed < 0:
            raise ValueError("Bias update speed must be non-negative, got {}".format(bias_update_speed))

        self.num_routed_experts = num_routed_experts
        self.top_k = top_k
        self.bias_update_speed = bias_update_speed

        # Router projection. External expert_bias handles balancing
        self.router = nn.Linear(d_model, num_routed_experts, bias=False)

        # Persistent routing bias used only for top-k expert selection
        self.register_buffer("expert_bias", torch.zeros(num_routed_experts))

        # Last observed batch load, used by update_bias()
        self.register_buffer("last_expert_load", torch.zeros(num_routed_experts), persistent=False)


    @torch.no_grad()
    def update_bias(self) -> None:
        """
        Update bias after a training step using:
            b_i <- b_i - gamma * sign(load_i - mean_load)

        Equivalent meaning:
            - overloaded expert -> decrease bias
            - underloaded expert -> increase bias
        """

        mean_load = self.last_expert_load.mean()
        error = self.last_expert_load - mean_load
        self.expert_bias.sub_(self.bias_update_speed * torch.sign(error))

    def forward(self, x: torch.Tensor) -> RouterStats:
        affinity_scores = torch.sigmoid(self.router(x)) # [B, T, num_routed_experts]

        # Bias is used only for routing selection.
        biased_scores = affinity_scores + self.expert_bias.view(1, 1, -1) # [B, T, num_routed_experts]

        # Select experts using biased scores for the selected experts
        _, top_k_indices = torch.topk(biased_scores, k=self.top_k, dim=-1) # [B, T, top_k]

        # gather original affinity scores for the selected experts
        selected_affinity = affinity_scores.gather(-1, top_k_indices) # [B, T, top_k]

        # Normalize the selected experts' affinity scores to get mixing weights
        denominator = selected_affinity.sum(dim=-1, keepdim=True).clamp_min(1e-9) # [B, T, 1]
        top_k_gates =  selected_affinity / denominator # [B, T, top_k]

        # Count token assignments per routed expert
        expert_load = F.one_hot(top_k_indices, num_classes=self.num_routed_experts).sum(dim=(0, 1, 2)).to(dtype=affinity_scores.dtype) # [num_routed_experts]

        if self.training:
            self.last_expert_load.copy_(expert_load.detach())

        return RouterStats(
            affinity_scores=affinity_scores,
            biased_scores=biased_scores,
            top_k_indices=top_k_indices,
            top_k_gates=top_k_gates,
            expert_load=expert_load,
            expert_bias=self.expert_bias.detach().clone()
        )


class MoEMLP(nn.Module):
    """
    MoE MLP:
        - shared experts: always active
        - routed experts: top-k selected with loss-free bias balancing

    This model return only MLP contribution.
    """
    def __init__(
            self,
            d_model: int,
            hidden_dim: int,
            num_shared_experts: int,
            num_routed_experts: int,
            top_k: int,
            dropout: float = 0.0,
            expert_linear_bias: bool = True,
            expert_bias_update_speed: float = 1e-3,
    ) -> None:
        super().__init__()

        self.shared_experts = SharedExperts(d_model, hidden_dim, num_shared_experts, dropout=dropout, bias=expert_linear_bias)

        self.routed_experts = nn.ModuleList([
            SwiGLUExpert(d_model, hidden_dim, dropout, bias=expert_linear_bias)
            for _ in range(num_routed_experts)
        ])

        self.router = LossFreeBiasRouter(d_model=d_model, num_routed_experts=num_routed_experts, top_k=top_k, bias_update_speed=expert_bias_update_speed)

        self.num_routed_experts = num_routed_experts
        self.top_k = top_k

    @torch.no_grad()
    def update_router_bias(self) -> None:
        """
        Call this function once after each training step
        """
        self.router.update_bias()

    def forward(
            self,
            x: torch.Tensor,
            return_router_stats: bool = False,
    ):
        """
        Args:
            x: [B, T, D]
            return_router_stats: whether to return routing statistics for analysis/debugging

        Returns:
            out: [B, T, D]
            (optional) router_stats: RouterStats dataclass containing routing information
        """

        if x.dim() != 3:
            raise ValueError(f"Input tensor must be 3D (B, T, D), got {x.shape}")

        batch_size, seq_len, d_model = x.shape

        # shared experts: always active
        shared_out = self.shared_experts(x) # [B, T, D]

        # routed experts: select top-k experts for each token
        stats = self.router(x) # RouterStats

        # Start with the shared contribution
        out = shared_out.clone()

        flat_x = x.reshape(-1, d_model) # [B*T, D]
        flat_out = out.reshape(-1, d_model) # [B*T, D]
        flat_indices = stats.top_k_indices.reshape(-1, self.top_k) # [B*T, top_k]
        flat_gates = stats.top_k_gates.reshape(-1, self.top_k) # [B*T, top_k]

        # dispatch tokens to each expert
        for expert_id, expert in enumerate(self.routed_experts):
            token_positions, kth_choice = torch.where(flat_indices == expert_id)

            if token_positions.numel() == 0:
                continue

            expert_input = flat_x[token_positions] # [num_tokens_for_expert, D]
            expert_output = expert(expert_input) # [num_tokens_for_expert, D]
            gates = flat_gates[token_positions, kth_choice].unsqueeze(-1)
            weighted_output = expert_output * gates.to(expert_output.dtype)

            flat_out.index_add_(0, token_positions, weighted_output)

        out = flat_out.view(batch_size, seq_len, d_model)

        if return_router_stats:
            return out, stats

        return out



### Transformer Block integrating all components

In [13]:
class RMSNorm(nn.Module):
    """
    Root Mean Square Layer Normalization (RMSNorm) without a mean subtraction
    """

    def __init__(self, dim: int, eps: float = 1e-8) -> None:
        super().__init__()

        if dim <= 0:
            raise ValueError(f"RMSNorm requires at least one dimension, got {dim}")

        if eps <= 0:
            raise ValueError(f"RMSNorm epsilon must be positive, got {eps}")

        self.weight = nn.Parameter(torch.ones(dim))
        self.eps = eps

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x: [..., D]

        Returns:
            out: [..., D]
        """

        rms = x.pow(2).mean(dim=-1, keepdim=True)
        x = x * torch.rsqrt(rms + self.eps)

        return x * self.weight


class TransformerBlock(nn.Module):
    """
    Transformer block:
        x = x + MLA(RMSNorm(x))
        x = x + MoEMLP(RMSNorm(x))

    Expected external modules:
        - MultiHeadLatentAttention
        - MoEMLP
        - RMSNorm

    Args:
        d_model: model width
        num_heads: number of attention heads
        head_dim: dimension of each attention head
        q_lora_rank: query latent rank in MLA
        KV_lora_rank: KV latent rank in MLA
        rope_head_dim: decoupled RoPE dim per head in MLA

        mlp_hidden_dim: hidden size inside each expert FFN
        num_shared_experts: number of shared experts
        num_routed_experts: number of routed experts
        top_k: number of experts to select per token

        attn_dropout: dropout rate for attention probabilities
        mlp_dropout: dropout rate for MLP experts
        resid_dropout: optional dropout on sublayer outputs before residual add
        norm_eps: RMSNorm epsilon
        bias: bias flag passed into projections/experts
        router_bias_update_speed: update speed for loss-free expert bias balancing
        return_router_stats: whether to return routing statistics for analysis/debugging
    """

    def __init__(
            self,
            d_model: int,
            num_heads: int,
            head_dim: int,
            q_lora_rank: int,
            kv_lora_rank: int,
            rope_head_dim: int,
            mlp_hidden_dim: int,
            num_shared_experts: int,
            num_routed_experts: int,
            top_k: int,
            attn_dropout: float = 0.0,
            mlp_dropout: float = 0.0,
            resid_dropout: float = 0.0,
            norm_eps: float = 1e-6,
            bias: bool = False,
            rope_base: int = 10_000,
            router_bias_update_speed: float = 1e-3,
                 ) -> None:
        super().__init__()

        self.attn_norm = RMSNorm(d_model, eps=norm_eps)
        self.ffn_norm = RMSNorm(d_model, eps=norm_eps)

        self.attn = MultiHeadLatentAttention(
            d_model=d_model,
            num_heads=num_heads,
            head_dim=head_dim,
            q_lora_rank=q_lora_rank,
            kv_lora_rank=kv_lora_rank,
            rope_head_dim=rope_head_dim,
            dropout=attn_dropout,
            bias=bias,
            rope_base=rope_base
        )

        self.mlp = MoEMLP(
            d_model=d_model,
            hidden_dim=mlp_hidden_dim,
            num_shared_experts=num_shared_experts,
            num_routed_experts=num_routed_experts,
            top_k=top_k,
            dropout=mlp_dropout,
            expert_linear_bias=bias,
            expert_bias_update_speed=router_bias_update_speed
        )

        self.resid_dropout = nn.Dropout(resid_dropout)

    @torch.no_grad()
    def update_router_bias(self) -> None:
        """
        Call this function once after optimizer.step() to update the router bias based on the last observed expert load.
        """
        self.mlp.update_router_bias()

    def forward(
            self,
            x: torch.Tensor,
            attention_mask: torch.Tensor | None = None,
            *,
            is_causal: bool = True,
            start_pos: int = 0,
            return_router_stats: bool = False,
            return_attn_weights: bool = False
                ):

        """
        Args:
            x: [B, T, D]
            attention_mask: [B, T] with 1/True for valid tokens
            is_causal: apply casual mask in attention
            start_pos: RoPE start offset for generation
            return_attn_weights: whether to return attention weights
            return_router_stats: whether to return routing statistics for analysis/debugging

        Returns:
            out: [B, T, D]
            (optional) attn_weights: [B, num_heads, T, T]
            (optional) router_stats: RouterStats dataclass containing routing information
        """
        # Attention sub-layer
        residual = x
        attn_input = self.attn_norm(x)

        if return_attn_weights:
            attn_out, attn_weights = self.attn(attn_input, attention_mask=attention_mask, is_casual=is_causal, start_pos=start_pos, return_attn_weights=True)
        else:
            attn_out = self.attn(attn_input, attention_mask=attention_mask, is_casual=is_causal, start_pos=start_pos, return_attn_weights=False)
            attn_weights = None

        x = residual + self.resid_dropout(attn_out)

        # MoE MLP sub-layer
        residual = x
        mlp_input = self.ffn_norm(x)

        if return_router_stats:
            mlp_out, router_stats = self.mlp(mlp_input, return_router_stats=True)
        else:
            mlp_out = self.mlp(mlp_input, return_router_stats=False)
            router_stats = None

        x = residual + self.resid_dropout(mlp_out)

        if not return_router_stats and not return_attn_weights:
            return x

        if return_attn_weights and not return_router_stats:
            return x, attn_weights

        if return_router_stats and not return_attn_weights:
            return x, router_stats

        return x, attn_weights, router_stats



## Implementation of the language model with Multi-token prediction capabilities

### Global model configuration and hyperparameters

In [14]:
@dataclass
class ModelConfig:
    # vocabulary / sequence
    vocab_size: int
    max_seq_len: int
    pad_token_id: int = 0

    # model width / depth
    d_model: int = 512
    num_layers: int = 8

    # MLA
    num_heads: int = 8
    head_dim: int = 64
    q_lora_rank: int = 128
    kv_lora_rank: int = 128
    rope_head_dim: int = 32
    rope_base: int = 10_000

    # MoE MLP
    mlp_hidden_dim: int = 2048
    num_shared_experts: int = 1
    num_routed_experts: int = 8
    top_k: int = 2
    router_bias_update_speed: float = 1e-3

    # dropout / norm / misc
    attn_dropout: float = 0.0
    mlp_dropout: float = 0.0
    resid_dropout: float = 0.0
    emb_dropout: float = 0.0
    norm_eps: float = 1e-6
    bias: bool = False
    tie_word_embeddings: bool = True

    # MTP
    mtp_depth: int = 0
    mtp_lambda: float = 1.0

    def __post_init__(self) -> None:
        if self.vocab_size <= 0:
            raise ValueError("vocab_size must be > 0")
        if self.max_seq_len <= 0:
            raise ValueError("max_seq_len must be > 0")
        if self.d_model <= 0:
            raise ValueError("d_model must be > 0")
        if self.num_layers <= 0:
            raise ValueError("num_layers must be > 0")
        if self.num_heads <= 0:
            raise ValueError("num_heads must be > 0")
        if self.head_dim <= 0:
            raise ValueError("head_dim must be > 0")
        if self.q_lora_rank <= 0:
            raise ValueError("q_lora_rank must be > 0")
        if self.kv_lora_rank <= 0:
            raise ValueError("kv_lora_rank must be > 0")
        if self.rope_head_dim <= 0 or self.rope_head_dim % 2 != 0:
            raise ValueError("rope_head_dim must be a positive even number")
        if self.mlp_hidden_dim <= 0:
            raise ValueError("mlp_hidden_dim must be > 0")
        if self.num_shared_experts < 0:
            raise ValueError("num_shared_experts must be >= 0")
        if self.num_routed_experts <= 0:
            raise ValueError("num_routed_experts must be > 0")
        if self.top_k <= 0 or self.top_k > self.num_routed_experts:
            raise ValueError("top_k must be in [1, num_routed_experts]")
        if self.mtp_depth < 0:
            raise ValueError("mtp_depth must be >= 0")
        if self.mtp_lambda < 0:
            raise ValueError("mtp_lambda must be >= 0")


In [15]:
@dataclass
class ModelOutput:
    logits: torch.Tensor
    hidden_states: torch.Tensor
    loss: torch.Tensor | None = None
    main_loss: torch.Tensor | None = None
    mtp_loss: torch.Tensor | None = None
    mtp_logits: torch.Tensor | None = None
    mtp_hidden_states: List[torch.Tensor] | None = None
    mtp_valid_masks: List[torch.Tensor] | None = None


In [16]:
class MTPModule(nn.Module):
    """
    One sequential MTP module
    h'_k = projection([RMSNorm(previous_hidden), RMSNorm(Emb(future_token))])
    h_k = TransformerBlock(h'_k)

    Shared embedding and shared LM head stay in the main model
    """

    def __init__(self, config: ModelConfig) -> None:
        super().__init__()
        self.prev_hidden_norm = RMSNorm(config.d_model, eps=config.norm_eps)
        self.future_emb_norm = RMSNorm(config.d_model, eps=config.norm_eps)

        self.proj = nn.Linear(2*config.d_model, config.d_model, bias=config.bias)
        self.block = TransformerBlock(d_model=config.d_model,
                                      num_heads=config.num_heads,
                                      head_dim=config.head_dim,
                                      q_lora_rank=config.q_lora_rank,
                                      kv_lora_rank=config.kv_lora_rank,
                                      rope_head_dim=config.rope_head_dim,
                                      mlp_hidden_dim=config.mlp_hidden_dim,
                                      num_shared_experts=config.num_shared_experts,
                                      num_routed_experts=config.num_routed_experts,
                                      top_k=config.top_k,
                                      attn_dropout=config.attn_dropout,
                                      mlp_dropout=config.mlp_dropout,
                                      resid_dropout=config.resid_dropout,
                                      norm_eps=config.norm_eps,
                                      bias=config.bias,
                                      rope_base=config.rope_base,
                                      router_bias_update_speed=config.router_bias_update_speed
                                     )

    def forward(
            self,
            prev_hidden: torch.Tensor,
            future_token_ids: torch.Tensor,
            embedding_layer: InputEmbeddings,
            attention_mask: torch.Tensor | None = None,
            *,
            start_pos: int = 0,
    ) -> torch.Tensor:
        future_emb =  embedding_layer(future_token_ids)
        fused = torch.cat([
            self.prev_hidden_norm(prev_hidden),
            self.future_emb_norm(future_emb),
            ],dim=-1) # [B, T, 2*D]


        fused = self.proj(fused)

        return self.block(
            fused,
            attention_mask=attention_mask,
            is_causal=True,
            start_pos=start_pos,
            return_attn_weights=False,
            return_router_stats=False
        )

    @torch.no_grad()
    def update_router_bias(self) -> None:
        self.block.update_router_bias()


### Language model implementation

In [17]:
class LanguageModel(nn.Module):
    def __init__(self, config: ModelConfig) -> None:
        super().__init__()
        self.config = config

        self.embedding = InputEmbeddings(
            vocab_size=config.vocab_size,
            d_model=config.d_model,
            padding_idx=config.pad_token_id,
            scale=True
        )

        self.emb_dropout = nn.Dropout(config.emb_dropout)
        self.blocks = nn.ModuleList(
            [
                TransformerBlock(
                    d_model=config.d_model,
                    num_heads=config.num_heads,
                    head_dim=config.head_dim,
                    q_lora_rank=config.q_lora_rank,
                    kv_lora_rank=config.kv_lora_rank,
                    rope_head_dim=config.rope_head_dim,
                    mlp_hidden_dim=config.mlp_hidden_dim,
                    num_shared_experts=config.num_shared_experts,
                    num_routed_experts=config.num_routed_experts,
                    top_k=config.top_k,
                    attn_dropout=config.attn_dropout,
                    mlp_dropout=config.mlp_dropout,
                    resid_dropout=config.resid_dropout,
                    norm_eps=config.norm_eps,
                    bias=config.bias,
                    rope_base=config.rope_base,
                    router_bias_update_speed=config.router_bias_update_speed
                )
                for _ in range(config.num_layers)
            ]
        )

        self.final_norm = RMSNorm(
            config.d_model,
            eps=config.norm_eps,
        )

        self.lm_head = nn.Linear(
            config.d_model,
            config.vocab_size,
            bias=False
        )

        if config.tie_word_embeddings:
            self.lm_head.weight = self.embedding.embedding.weight

        self.mtp_modules = nn.ModuleList(
            [
                MTPModule(config)
                for _ in range(config.mtp_depth)
            ]
        )

    def _forward_backbone(
            self,
            input_ids: torch.Tensor,
            attention_mask: torch.Tensor | None = None,
            *,
            start_pos: int = 0,
    ) -> torch.Tensor:
        if input_ids.dim() != 2:
            raise ValueError(f"input_ids must be of shape (B, T), got {input_ids.shape}")

        _, seq_len = input_ids.shape
        if seq_len > self.config.max_seq_len:
            raise ValueError(f"Sequence length {seq_len} exceeds model maximum of {self.config.max_seq_len}")

        if attention_mask is None:
            attention_mask = (input_ids != self.config.pad_token_id)

        x = self.embedding(input_ids)
        x = self.emb_dropout(x)

        for block in self.blocks:
            x = block(
                x,
                attention_mask=attention_mask,
                is_causal=True,
                start_pos=start_pos,
                return_attn_weights=False,
                return_router_stats=False
            )

        return self.final_norm(x)

    def _forward_mtp(
            self,
            input_ids: torch.Tensor,
            attention_mask: torch.Tensor,
            *,
            backbone_hidden_states: torch.Tensor,
            labels: torch.Tensor | None = None,
            start_pos: int = 0
    ):
        """
        Vectorized MTP across sequence, sequential across depth.

        Uses the same chaining idea as the simple file:
            prev_hidden -> depth 1 -> depth 2 -> ... -> depth K -> MTP prediction

        but without looping over sequence positions one by one.

        Args:
            input_ids: [B, T]
            attention_mask: [B, T]
            backbone_hidden: [B, T, D]
            labels: [B, T] with future token ids for MTP prediction (shifted by 1 compared to input_ids)
            start_pos: RoPE start offset for generation
        """

        mtp_logits_list: List[torch.Tensor] = []
        mtp_hidden_list: List[torch.Tensor] = []
        mtp_valid_masks: List[torch.Tensor] = []
        mtp_losses: List[torch.Tensor] = []

        prev_hidden = backbone_hidden_states
        _, seq_len, _ = backbone_hidden_states.shape

        for depth_idx, mtp_module in enumerate(self.mtp_modules, start=1):
            # depth_idx = 1 predicts +2 token, depth_idx = 2 predicts +3 token, etc.
            current_len = seq_len -  (depth_idx + 1)

            if current_len <= 0:
                break

            prev_hidden_slice = prev_hidden[:,:current_len,:] # [B, T - (depth_idx + 1), D]
            # input future token for this depth: t_{i+k}
            future_token_ids = input_ids[:, depth_idx:depth_idx + current_len] # [B, T - (depth_idx + 1)]

            # position is valid only if:
            # current token exists, future input token exists, and future target token exists
            valid_mask = (
                attention_mask[:, :current_len]
                & attention_mask[:, depth_idx: current_len + depth_idx]
                & attention_mask[:, depth_idx + 1: current_len + depth_idx + 1]
            ) # [B, T - (depth_idx + 1)]

            h_k = mtp_module(
                prev_hidden=prev_hidden_slice,
                future_token_ids=future_token_ids,
                embedding_layer=self.embedding,
                attention_mask=valid_mask,
                start_pos=start_pos,
            )

            h_k = self.final_norm(h_k)
            logits_k = self.lm_head(h_k)

            mtp_hidden_list.append(h_k)
            mtp_logits_list.append(logits_k)
            mtp_valid_masks.append(valid_mask)

            if labels is not None:
                target_k = labels[:, depth_idx+1: depth_idx+1+current_len].clone() # [B, T - (depth_idx + 1)]

                # ignore invalid padded positions in loss
                target_k = target_k.masked_fill(~valid_mask, -100)

                loss_k = F.cross_entropy(logits_k.reshape(-1, self.config.vocab_size), target_k.reshape(-1), ignore_index=-100)
                mtp_losses.append(loss_k)

            prev_hidden = h_k

        mtp_loss = None
        if mtp_losses:
            mtp_loss = self.config.mtp_lambda * torch.stack(mtp_losses).mean()


        return mtp_logits_list, mtp_hidden_list, mtp_valid_masks, mtp_loss

    def forward(
            self,
            input_ids: torch.Tensor,
            attention_mask: torch.Tensor | None = None,
            labels: torch.Tensor | None = None,
            *,
            start_pos: int = 0,
            return_mtp: bool = True,
            backbone_hidden_states: torch.Tensor | None = None
    ) -> ModelOutput:

        if attention_mask is None:
            attention_mask = (input_ids != self.config.pad_token_id)

        if backbone_hidden_states is None:
            hidden_states = self._forward_backbone(
                input_ids=input_ids,
                attention_mask=attention_mask,
                start_pos=start_pos
            )

        else:
            hidden_states = backbone_hidden_states


        logits = self.lm_head(hidden_states)

        main_loss = None
        if labels is not None:
            if labels.shape != input_ids.shape:
                raise ValueError(f"Labels shape {labels.shape} must match input_ids shape {input_ids.shape}")

            main_targets = labels[:, 1:].clone()
            main_targets = main_targets.masked_fill(~attention_mask[:, 1:], -100)

            main_loss = F.cross_entropy(
                logits[:,:-1,:].reshape(-1, self.config.vocab_size),
                main_targets.reshape(-1),
                ignore_index=-100
            )

        mtp_logits = None
        mtp_hidden_states = None
        mtp_valid_masks = None
        mtp_loss = None

        if return_mtp and self.config.mtp_depth > 0:
            mtp_logits, mtp_hidden_states, mtp_valid_masks, mtp_loss = self._forward_mtp(
                input_ids=input_ids,
                attention_mask=attention_mask,
                backbone_hidden_states=hidden_states,
                labels=labels,
                start_pos=start_pos
            )

        total_loss = None

        if main_loss is not None and mtp_loss is not None:
            total_loss = main_loss + mtp_loss
        elif main_loss is not None:
            total_loss = main_loss
        elif mtp_loss is not None:
            total_loss = mtp_loss

        return ModelOutput(
            logits=logits,
            hidden_states=hidden_states,
            loss=total_loss,
            main_loss=main_loss,
            mtp_loss=mtp_loss,
            mtp_logits=mtp_logits if mtp_logits is not None else None,
            mtp_hidden_states=mtp_hidden_states if mtp_hidden_states is not None else None,
            mtp_valid_masks=mtp_valid_masks if mtp_valid_masks is not None else None
        )

    @torch.no_grad()
    def update_router_biases(self) -> None:
        for block in self.blocks:
            block.update_router_bias()

        for mtp_module in self.mtp_modules:
            mtp_module.update_router_bias()

In [18]:
# ============================================================
# Paths (anchored to the project root so saves always land in the right place)
# ============================================================
_PROJECT_ROOT = Path.cwd()
print(_PROJECT_ROOT)
MODELS_DIR = _PROJECT_ROOT / "models"
SAVED_DIR  = _PROJECT_ROOT / "saved"
DATA_DIR   = _PROJECT_ROOT / "data"
print(MODELS_DIR)
print(SAVED_DIR)
print(DATA_DIR)
MODELS_DIR.mkdir(parents=True, exist_ok=True)
SAVED_DIR.mkdir(parents=True, exist_ok=True)
DATA_DIR.mkdir(parents=True, exist_ok=True)

BEST_MODEL_PATH = MODELS_DIR / "best_model.pt"
LAST_CHECKPOINT_PATH = MODELS_DIR / "last_checkpoint.pt"
CONFIG_PATH = MODELS_DIR / "config.json"
TRAIN_PLOT_PATH = SAVED_DIR / "training_curve.png"
TRAIN_HISTORY_PATH = SAVED_DIR / "training_history.json"

# Existing notebook-style tokenizer path
TOKENIZER_PATH = DATA_DIR / "bpe_tokenizer.json"


/Users/azire/PycharmProjects/DeepSeek R1 Based LLM
/Users/azire/PycharmProjects/DeepSeek R1 Based LLM/models
/Users/azire/PycharmProjects/DeepSeek R1 Based LLM/saved
/Users/azire/PycharmProjects/DeepSeek R1 Based LLM/data


In [19]:

# ============================================================
# Tokenizer helpers
# ============================================================
def load_bpe_tokenizer(tokenizer_path: Union[str, Path] = TOKENIZER_PATH) -> Tokenizer:
    tokenizer_path = Path(tokenizer_path)
    if not tokenizer_path.exists():
        raise FileNotFoundError(
            f"BPE tokenizer file not found at {tokenizer_path}. "
            f"Your notebook trained and saved it there, so save the tokenizer first."
        )
    return Tokenizer.from_file(str(tokenizer_path))


def encode_text(tokenizer: Tokenizer, text: str) -> List[int]:
    return tokenizer.encode(text).ids


def decode_ids(tokenizer: Tokenizer, ids: List[int]) -> str:
    return tokenizer.decode(ids, skip_special_tokens=True)


def get_pad_token_id(tokenizer: Tokenizer) -> int:
    pad_id = tokenizer.token_to_id("[PAD]")
    if pad_id is None:
        raise ValueError("Tokenizer does not contain [PAD] token.")
    return pad_id


def get_vocab_size(tokenizer: Tokenizer) -> int:
    return tokenizer.get_vocab_size()


In [20]:

# ============================================================
# Dataset
# ============================================================
class TokenSequenceDataset(Dataset):
    """
    Creates fixed-length causal LM sequences from already-tokenized IDs.

    Returns:
        input_ids:      [seq_len]
        labels:         [seq_len]
        attention_mask: [seq_len]
    """
    def __init__(self, token_ids: List[int], seq_len: int) -> None:
        if seq_len < 2:
            raise ValueError("seq_len must be at least 2")
        if len(token_ids) <= seq_len:
            raise ValueError(
                f"Not enough tokens for seq_len={seq_len}. "
                f"Need > {seq_len}, got {len(token_ids)}."
            )
        self.token_ids = token_ids
        self.seq_len = seq_len

    def __len__(self) -> int:
        # We use labels=input_ids clone, and the model shifts internally.
        return len(self.token_ids) - self.seq_len

    def __getitem__(self, idx: int) -> Dict[str, torch.Tensor]:
        chunk = self.token_ids[idx: idx + self.seq_len]
        input_ids = torch.tensor(chunk, dtype=torch.long)
        labels = input_ids.clone()
        attention_mask = torch.ones_like(input_ids, dtype=torch.bool)
        return {
            "input_ids": input_ids,
            "labels": labels,
            "attention_mask": attention_mask,
        }


In [21]:

# ============================================================
# Utilities
# ============================================================
def save_json(path: Path, data: Dict[str, Any]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(data, ensure_ascii=False, indent=2), encoding="utf-8")


def plot_training_curves(history: Dict[str, List[float]], save_path: Path) -> None:
    save_path.parent.mkdir(parents=True, exist_ok=True)
    plt.figure(figsize=(8, 5))
    if history.get("train_loss"):
        plt.plot(history["train_loss"], label="train_loss")
    if history.get("val_loss"):
        plt.plot(history["val_loss"], label="val_loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title("Training curve")
    plt.legend()
    plt.tight_layout()
    plt.savefig(save_path)
    plt.close()


def save_checkpoint(
    *,
    checkpoint_path: Path,
    model: nn.Module,
    optimizer: torch.optim.Optimizer,
    epoch: int,
    best_val_loss: float,
    history: Dict[str, List[float]],
    config: ModelConfig,
) -> None:
    payload = {
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "best_val_loss": best_val_loss,
        "history": history,
        "config": asdict(config),
    }
    checkpoint_path.parent.mkdir(parents=True, exist_ok=True)
    torch.save(payload, checkpoint_path)


def save_best_model(
    *,
    path: Path,
    model: nn.Module,
    epoch: int,
    val_loss: float,
    config: ModelConfig,
) -> None:
    payload = {
        "epoch": epoch,
        "val_loss": val_loss,
        "model_state_dict": model.state_dict(),
        "config": asdict(config),
    }
    path.parent.mkdir(parents=True, exist_ok=True)
    torch.save(payload, path)


def load_model_from_checkpoint(
    *,
    model_class,
    checkpoint_path: Path,
    device: torch.device,
) -> Tuple[nn.Module, Dict[str, Any]]:
    checkpoint = torch.load(checkpoint_path, map_location=device)
    config = ModelConfig(**checkpoint["config"])
    model = model_class(config)
    model.load_state_dict(checkpoint["model_state_dict"])
    model.to(device)
    return model, checkpoint


def load_or_create_model(
    *,
    model_class,
    config: ModelConfig,
    device: torch.device,
    prefer_best: bool = False,
) -> Tuple[nn.Module, Optional[Dict[str, Any]], Path]:
    if prefer_best and BEST_MODEL_PATH.exists():
        model, checkpoint = load_model_from_checkpoint(
            model_class=model_class,
            checkpoint_path=BEST_MODEL_PATH,
            device=device,
        )
        return model, checkpoint, BEST_MODEL_PATH

    if LAST_CHECKPOINT_PATH.exists():
        model, checkpoint = load_model_from_checkpoint(
            model_class=model_class,
            checkpoint_path=LAST_CHECKPOINT_PATH,
            device=device,
        )
        return model, checkpoint, LAST_CHECKPOINT_PATH

    if BEST_MODEL_PATH.exists():
        model, checkpoint = load_model_from_checkpoint(
            model_class=model_class,
            checkpoint_path=BEST_MODEL_PATH,
            device=device,
        )
        return model, checkpoint, BEST_MODEL_PATH

    model = model_class(config).to(device)
    return model, None, Path("")


In [22]:

# ============================================================
# Training / evaluation
# ============================================================
def run_one_epoch(
    *,
    model: LanguageModel,
    dataloader: DataLoader,
    device: torch.device,
    optimizer: Optional[torch.optim.Optimizer] = None,
    grad_clip: Optional[float] = 1.0,
) -> float:
    is_train = optimizer is not None
    model.train(is_train)

    total_loss = 0.0
    total_steps = 0

    for batch in dataloader:
        input_ids = batch["input_ids"].to(device)
        labels = batch["labels"].to(device)
        attention_mask = batch["attention_mask"].to(device)

        with torch.set_grad_enabled(is_train):
            output = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=labels,
                return_mtp=True,
            )
            loss = output.loss
            if loss is None:
                raise RuntimeError("Model returned loss=None during training/evaluation.")

            if is_train:
                optimizer.zero_grad(set_to_none=True)
                loss.backward()
                if grad_clip is not None and grad_clip > 0:
                    torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
                optimizer.step()

                # MoE router balancing update, if your model has it
                if hasattr(model, "update_router_biases"):
                    model.update_router_biases()

        total_loss += float(loss.detach().item())
        total_steps += 1

    if total_steps == 0:
        return float("inf")
    return total_loss / total_steps


In [23]:

# ============================================================
# Training
# ============================================================
def train_model(
    *,
    train_text: str,
    val_text: str,
    tokenizer_path: Union[str, Path] = TOKENIZER_PATH,
    config: Optional[ModelConfig] = None,
    seq_len: int = 128,
    batch_size: int = 16,
    epochs: int = 5,
    lr: float = 3e-4,
    weight_decay: float = 0.01,
    device: Optional[str] = None,
) -> LanguageModel:
    device_obj = torch.device(device or ("cuda" if torch.cuda.is_available() else "cpu"))
    tokenizer = load_bpe_tokenizer(tokenizer_path)

    vocab_size = get_vocab_size(tokenizer)
    pad_token_id = get_pad_token_id(tokenizer)

    # Prefer checkpoint config if it exists
    if LAST_CHECKPOINT_PATH.exists():
        checkpoint = torch.load(LAST_CHECKPOINT_PATH, map_location="cpu")
        config = ModelConfig(**checkpoint["config"])
    elif BEST_MODEL_PATH.exists():
        checkpoint = torch.load(BEST_MODEL_PATH, map_location="cpu")
        config = ModelConfig(**checkpoint["config"])
    else:
        if config is None:
            config = ModelConfig(
                vocab_size=vocab_size,
                max_seq_len=seq_len,
                pad_token_id=pad_token_id,
                d_model=256,
                num_layers=4,
                num_heads=4,
                head_dim=64,
                q_lora_rank=64,
                kv_lora_rank=64,
                rope_head_dim=32,
                mlp_hidden_dim=512,
                num_shared_experts=1,
                num_routed_experts=4,
                top_k=2,
                mtp_depth=2,
                mtp_lambda=1.0,
            )
        else:
            config.vocab_size = vocab_size
            config.max_seq_len = seq_len
            config.pad_token_id = pad_token_id

    save_json(CONFIG_PATH, asdict(config))

    model, resume_checkpoint, loaded_from = load_or_create_model(
        model_class=LanguageModel,
        config=config,
        device=device_obj,
        prefer_best=False,
    )

    optimizer = AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)

    start_epoch = 0
    best_val_loss = float("inf")
    history = {"train_loss": [], "val_loss": []}

    if resume_checkpoint is not None:
        start_epoch = int(resume_checkpoint.get("epoch", 0)) + 1
        best_val_loss = float(resume_checkpoint.get("best_val_loss", float("inf")))
        history = resume_checkpoint.get("history", history)
        if "optimizer_state_dict" in resume_checkpoint:
            optimizer.load_state_dict(resume_checkpoint["optimizer_state_dict"])
        print(f"Resumed from: {loaded_from}")

    train_ids = encode_text(tokenizer, train_text)
    val_ids = encode_text(tokenizer, val_text)

    train_dataset = TokenSequenceDataset(train_ids, seq_len=seq_len)
    val_dataset = TokenSequenceDataset(val_ids, seq_len=seq_len)

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

    for epoch in range(start_epoch, epochs):
        train_loss = run_one_epoch(
            model=model,
            dataloader=train_loader,
            device=device_obj,
            optimizer=optimizer,
            grad_clip=1.0,
        )

        val_loss = run_one_epoch(
            model=model,
            dataloader=val_loader,
            device=device_obj,
            optimizer=None,
        )

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)

        print(
            f"Epoch {epoch + 1}/{epochs} | "
            f"train_loss={train_loss:.4f} | val_loss={val_loss:.4f}"
        )

        save_checkpoint(
            checkpoint_path=LAST_CHECKPOINT_PATH,
            model=model,
            optimizer=optimizer,
            epoch=epoch,
            best_val_loss=min(best_val_loss, val_loss),
            history=history,
            config=config,
        )

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            save_best_model(
                path=BEST_MODEL_PATH,
                model=model,
                epoch=epoch,
                val_loss=val_loss,
                config=config,
            )
            print(f"Saved new best model to: {BEST_MODEL_PATH}")

        plot_training_curves(history, TRAIN_PLOT_PATH)
        save_json(TRAIN_HISTORY_PATH, history)

    return model

In [24]:


# ============================================================
# Inference
# ============================================================
@torch.no_grad()
def generate_text_with_mtp(
    *,
    prompt: str,
    tokenizer_path: Union[str, Path] = TOKENIZER_PATH,
    max_new_tokens: int = 50,
    device: Optional[str] = None,
    use_best: bool = True,
    do_sample: bool = False,
    temperature: float = 1.0,
) -> str:
    device_obj = torch.device(device or ("cuda" if torch.cuda.is_available() else "cpu"))
    tokenizer = load_bpe_tokenizer(tokenizer_path)
    vocab_size = get_vocab_size(tokenizer)
    pad_token_id = get_pad_token_id(tokenizer)

    model, checkpoint, loaded_from = load_or_create_model(
        model_class=LanguageModel,
        config=ModelConfig(
            vocab_size=vocab_size,
            max_seq_len=128,
            pad_token_id=pad_token_id,
        ),
        device=device_obj,
        prefer_best=use_best,
    )

    if checkpoint is None:
        raise FileNotFoundError("No saved model checkpoint found in models/")

    print(f"Loaded model from: {loaded_from}")
    model.eval()

    generated = list(encode_text(tokenizer, prompt))

    for _ in range(max_new_tokens):
        context = generated[-model.config.max_seq_len:]
        input_ids = torch.tensor([context], dtype=torch.long, device=device_obj)
        attention_mask = torch.ones_like(input_ids, dtype=torch.bool)

        # If your model has MTP draft helper, use it.
        if hasattr(model, "draft_next_tokens"):
            drafted = model.draft_next_tokens(
                input_ids=input_ids,
                attention_mask=attention_mask,
                depth=model.config.mtp_depth,
                do_sample=do_sample,
                temperature=temperature,
            )
            next_token_id = int(drafted[0, 0].item())
        else:
            output = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=None,
                return_mtp=True,
            )
            logits = output.logits[:, -1, :]
            if do_sample:
                probs = torch.softmax(logits / temperature, dim=-1)
                next_token_id = int(torch.multinomial(probs, num_samples=1)[0, 0].item())
            else:
                next_token_id = int(torch.argmax(logits[0], dim=-1).item())

        generated.append(next_token_id)

    return decode_ids(tokenizer, generated)


@torch.no_grad()
def inspect_mtp_predictions(
    *,
    prompt: str,
    tokenizer_path: Union[str, Path] = TOKENIZER_PATH,
    device: Optional[str] = None,
) -> Dict[str, Any]:
    device_obj = torch.device(device or ("cuda" if torch.cuda.is_available() else "cpu"))
    tokenizer = load_bpe_tokenizer(tokenizer_path)
    vocab_size = get_vocab_size(tokenizer)
    pad_token_id = get_pad_token_id(tokenizer)

    model, checkpoint, loaded_from = load_or_create_model(
        model_class=LanguageModel,
        config=ModelConfig(
            vocab_size=vocab_size,
            max_seq_len=128,
            pad_token_id=pad_token_id,
        ),
        device=device_obj,
        prefer_best=True,
    )

    if checkpoint is None:
        raise FileNotFoundError("No saved model checkpoint found in models/")

    print(f"Loaded model from: {loaded_from}")
    model.eval()

    prompt_ids = encode_text(tokenizer, prompt)
    input_ids = torch.tensor([prompt_ids], dtype=torch.long, device=device_obj)
    attention_mask = torch.ones_like(input_ids, dtype=torch.bool)

    output = model(
        input_ids=input_ids,
        attention_mask=attention_mask,
        labels=None,
        return_mtp=True,
    )

    main_next_token_id = int(torch.argmax(output.logits[0, -1], dim=-1).item())

    drafted_ids: Optional[List[int]] = None
    drafted_tokens: Optional[List[str]] = None
    if hasattr(model, "draft_next_tokens"):
        drafted_ids = model.draft_next_tokens(
            input_ids=input_ids,
            attention_mask=attention_mask,
            depth=model.config.mtp_depth,
            do_sample=False,
            temperature=1.0,
        )[0].tolist()
        drafted_tokens = [decode_ids(tokenizer, [idx]) for idx in drafted_ids]

    return {
        "prompt": prompt,
        "prompt_ids": prompt_ids,
        "main_next_token_id": main_next_token_id,
        "main_next_token": decode_ids(tokenizer, [main_next_token_id]),
        "draft_token_ids": drafted_ids,
        "draft_tokens": drafted_tokens,
    }


def main() -> None:
    """
    Small example with your existing BPE tokenizer.

    Before running:
      1) make sure your notebook already saved data/bpe_tokenizer.json
      2) replace 'your_model_module' import above
      3) ensure your model module exposes ModelConfig + LanguageModel
    """
    train_text = (
        "hello world. this is a tiny training example for a language model. "
        "multi token prediction helps the model learn future tokens. "
        "mixture of experts and mla are used inside the transformer blocks. "
    ) * 30

    val_text = (
        "hello world. this is a validation example. "
        "mtp can also be inspected during inference. "
    ) * 10

    tokenizer = load_bpe_tokenizer(TOKENIZER_PATH)
    config = ModelConfig(
        vocab_size=get_vocab_size(tokenizer),
        max_seq_len=128,
        pad_token_id=get_pad_token_id(tokenizer),
        d_model=128,
        num_layers=2,
        num_heads=4,
        head_dim=32,
        q_lora_rank=32,
        kv_lora_rank=32,
        rope_head_dim=16,
        mlp_hidden_dim=256,
        num_shared_experts=1,
        num_routed_experts=4,
        top_k=2,
        mtp_depth=2,
        mtp_lambda=1.0,
    )

    train_model(
        train_text=train_text,
        val_text=val_text,
        tokenizer_path=TOKENIZER_PATH,
        config=config,
        seq_len=128,
        batch_size=8,
        epochs=5,
        lr=3e-4,
    )

    generated = generate_text_with_mtp(
        prompt="hello world",
        tokenizer_path=TOKENIZER_PATH,
        max_new_tokens=30,
        use_best=True,
        do_sample=False,
    )
    print("\nGenerated text:")
    print(generated)

    mtp_info = inspect_mtp_predictions(
        prompt="hello world",
        tokenizer_path=TOKENIZER_PATH,
    )
    print("\nMTP inspection:")
    for key, value in mtp_info.items():
        print(f"{key}: {value}")


if __name__ == "__main__":
    main()


Resumed from: /Users/azire/PycharmProjects/DeepSeek R1 Based LLM/models/last_checkpoint.pt
Loaded model from: /Users/azire/PycharmProjects/DeepSeek R1 Based LLM/models/best_model.pt

Generated text:
he ll o world . this is a t in y tra in ing ex am ple for a l ang u age mo de l . m ul ti to ken pre di
Loaded model from: /Users/azire/PycharmProjects/DeepSeek R1 Based LLM/models/best_model.pt

MTP inspection:
prompt: hello world
prompt_ids: [84, 80, 55, 600]
main_next_token_id: 10
main_next_token: .
draft_token_ids: None
draft_tokens: None


In [35]:
from pathlib import Path
import os
import shutil
import torch
from torchview import draw_graph

# --------------------------------------------------
# Make sure Graphviz executable is visible
# --------------------------------------------------
os.environ["PATH"] += os.pathsep + "/opt/homebrew/bin"
os.environ["PATH"] += os.pathsep + "/usr/local/bin"

dot_path = shutil.which("dot")
print("dot found at:", dot_path)

if dot_path is None:
    raise RuntimeError(
        "Graphviz 'dot' executable was not found. "
        "Install it with 'brew install graphviz' and restart the kernel."
    )

# --------------------------------------------------
# Paths
# --------------------------------------------------
PROJECT_ROOT = Path.cwd()
MODELS_DIR = PROJECT_ROOT / "models"
SAVED_DIR = PROJECT_ROOT / "saved"

MODELS_DIR.mkdir(parents=True, exist_ok=True)
SAVED_DIR.mkdir(parents=True, exist_ok=True)

BEST_MODEL_PATH = MODELS_DIR / "best_model.pt"

# --------------------------------------------------
# Build model
# --------------------------------------------------
config = ModelConfig(
    vocab_size=1056,
    max_seq_len=128,
    pad_token_id=tokenizer.token_to_id("[PAD]"),
    d_model=128,
    num_layers=2,
    num_heads=4,
    head_dim=32,
    q_lora_rank=32,
    kv_lora_rank=32,
    rope_head_dim=16,
    mlp_hidden_dim=256,
    num_shared_experts=1,
    num_routed_experts=4,
    top_k=2,
    mtp_depth=2,
)

model = LanguageModel(config)

if BEST_MODEL_PATH.exists():
    checkpoint = torch.load(BEST_MODEL_PATH, map_location="cpu")
    state_dict = checkpoint["model_state_dict"] if "model_state_dict" in checkpoint else checkpoint
    model.load_state_dict(state_dict)
    print(f"Loaded model weights from: {BEST_MODEL_PATH}")
else:
    print("No best_model.pt found. Visualizing current model instance.")

model.eval().cpu()

# --------------------------------------------------
# Dummy input
# --------------------------------------------------
batch_size = 2
seq_len = min(16, config.max_seq_len)

input_ids = torch.randint(
    low=0,
    high=config.vocab_size,
    size=(batch_size, seq_len),
    dtype=torch.long,
)

attention_mask = torch.ones((batch_size, seq_len), dtype=torch.bool)

# --------------------------------------------------
# Draw and save
# --------------------------------------------------
graph = draw_graph(
    model,
    input_data={
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": None,
        "return_mtp": True,
    },
    graph_name="language_model_architecture",
    depth=6,
    expand_nested=True,
    hide_inner_tensors=True,
    hide_module_functions=True,
    show_shapes=True,
    save_graph=True,
    filename="language_model_architecture",
    directory=str(SAVED_DIR),
)

print(f"Saved architecture image to: {SAVED_DIR / 'language_model_architecture.png'}")
print(f"Saved graph source to: {SAVED_DIR / 'language_model_architecture.dot'}")

dot found at: /opt/homebrew/bin/dot
Loaded model weights from: /Users/azire/PycharmProjects/DeepSeek R1 Based LLM/models/best_model.pt
Saved architecture image to: /Users/azire/PycharmProjects/DeepSeek R1 Based LLM/saved/language_model_architecture.png
Saved graph source to: /Users/azire/PycharmProjects/DeepSeek R1 Based LLM/saved/language_model_architecture.dot
